# Full Initialization Workflow - Package

This notebook initializes a dynamic Modelica model that belongs to a package.

It takes the input package from `models/`, creates the auxiliary package, uses it to compute the initialization values, and generates the initialized package in `outputs/`.

In [ ]:
import logging
import os
import shutil
import sys

sys.path.append(os.path.abspath(".."))

logging.getLogger("OMPython").setLevel(logging.ERROR)

from OMPython import ModelicaSystemOMC, OMCSessionLocal

from scripts.helpers import *

In [ ]:
# User configuration: edit the values below for your package.

# Name of the package root model to initialize.
MODEL = "Demo_IEEE14.IEEE14DisconnectLine"

# Modelica Standard Library.
OMLIB_DIR = os.path.join(os.path.expanduser("~"), ".openmodelica", "libraries")
os.environ["OPENMODELICALIBRARY"] = OMLIB_DIR
MODELICA_PKG_PATH = os.path.join(OMLIB_DIR, "Modelica 3.2.3+maint.om", "package.mo")

# Dynawo Modelica library, for now the one the Julia family downloads.
DYNAWO_PKG_PATH = os.path.abspath(
    os.path.join("..", "..", "julia_openmodelica", "dynawo_library", "Dynawo", "package.mo")
)

# Optional settings

# INIT model selection for components with multiple INIT profiles.
INIT_MODEL_BY_COMPONENT = {}

# Leave empty to disable slack-specific handling.
SLACK_COMPONENT = "Gen1"

In [ ]:
# Internal paths derived from the user configuration above.
SOURCE_PACKAGE = MODEL.split(".")[0]
MODEL_DIR = os.path.abspath(os.path.join("models", SOURCE_PACKAGE))
OUTPUT_DIR = os.path.abspath("outputs")

PATHS = package_workflow_paths(MODEL, MODEL_DIR, OUTPUT_DIR)

MODELS_PKG_PATH = PATHS["source_package_file"]

AUX_PACKAGE = PATHS["aux_package"]
AUX_DIR = PATHS["aux_dir"]
AUX_ROOT_MODEL = PATHS["aux_root_model"]
AUX_PACKAGE_FILE = PATHS["aux_package_file"]
AUX_ORDER_FILE = PATHS["aux_order_file"]

INITIALIZED_PACKAGE = PATHS["initialized_package"]
INITIALIZED_DIR = PATHS["initialized_dir"]
INITIALIZED_ROOT_MODEL = PATHS["initialized_root_model"]
INITIALIZED_PACKAGE_FILE = PATHS["initialized_package_file"]
INITIALIZED_ORDER_FILE = PATHS["initialized_order_file"]

if not os.path.isfile(MODELS_PKG_PATH):
    raise RuntimeError(f"Package file not found: {MODELS_PKG_PATH}")
if not os.path.isfile(DYNAWO_PKG_PATH):
    raise RuntimeError(f"Dynawo package not found: {DYNAWO_PKG_PATH}")
if not os.path.isfile(MODELICA_PKG_PATH):
    raise RuntimeError(f"Modelica package not found: {MODELICA_PKG_PATH}")
if not os.path.isdir(OUTPUT_DIR):
    raise RuntimeError(f"Output directory not found: {OUTPUT_DIR}")

## Build Auxiliary Package

The original package root model is loaded and checked before any changes are made. An auxiliary package is then generated and saved for the initialization simulation.

In [ ]:
DynamicOMC = OMCSessionLocal()
omc_call(DynamicOMC, f'loadFile("{MODELICA_PKG_PATH}")')
omc_call(DynamicOMC, "loadModel(Complex)")
omc_call(DynamicOMC, "loadModel(ModelicaServices)")
omc_call(DynamicOMC, f'loadFile("{DYNAWO_PKG_PATH}")')
omc_call(DynamicOMC, f'loadFile("{MODELS_PKG_PATH}")')

check_user_configuration_package(
    DynamicOMC,
    model=MODEL,
    slack_component=SLACK_COMPONENT,
    init_model_by_component=INIT_MODEL_BY_COMPONENT,
)

In [ ]:
chain = get_inheritance_chain(DynamicOMC, MODEL)
AUX_NAME_MAP = auxiliary_name_map(chain, AUX_PACKAGE)

package_context = collect_package_component_contexts(DynamicOMC, chain)
components_by_model = package_context["components_by_model"]
patch_components_by_model = package_context["patch_components_by_model"]
global_blacklist_names = package_context["global_blacklist_names"]

# Drop any class left over from a previous run. A missing class is fine here.
send_expression(DynamicOMC, f"deleteClass({AUX_PACKAGE})")
omc_call(DynamicOMC, f'loadString("within ; package {AUX_PACKAGE} end {AUX_PACKAGE};")', parsed=False)

for model in chain:
    aux_model = AUX_NAME_MAP[model]
    aux_name = aux_model.split(".")[-1]

    print(f"Creating auxiliary model {aux_model} from {model}")
    omc_call(DynamicOMC, f'copyClass({model}, "{aux_name}", {AUX_PACKAGE})')

    components = components_by_model[model]

    apply_replacements(
        DynamicOMC,
        model,
        aux_model,
        components,
        SLACK_COMPONENT,
    )

    delete_connections(DynamicOMC, aux_model, components, global_targets=global_blacklist_names)
    delete_components(DynamicOMC, aux_model, components)

    add_init_models(
        DynamicOMC,
        model,
        aux_model,
        components,
        INIT_MODEL_BY_COMPONENT,
        SLACK_COMPONENT,
    )

    apply_LF_modifiers(
        DynamicOMC,
        model,
        aux_model,
        components,
    )

    add_init_equations(
        DynamicOMC,
        model,
        aux_model,
        components,
        INIT_MODEL_BY_COMPONENT,
        SLACK_COMPONENT,
    )

if os.path.isdir(AUX_DIR):
    shutil.rmtree(AUX_DIR)
os.makedirs(AUX_DIR, exist_ok=True)

write_package_files(
    AUX_PACKAGE_FILE,
    AUX_ORDER_FILE,
    AUX_PACKAGE,
    package_class_names(chain, AUX_NAME_MAP),
)

save_auxiliary_package_classes(
    DynamicOMC,
    chain,
    AUX_NAME_MAP,
    AUX_DIR,
    patch_components_by_model,
    SLACK_COMPONENT,
)

omc_call(DynamicOMC, f"deleteClass({AUX_PACKAGE})")
omc_call(DynamicOMC, f'loadFile("{AUX_PACKAGE_FILE}")')
omc_call(DynamicOMC, f"checkModel({AUX_ROOT_MODEL})", parsed=False)

print("Wrote auxiliary package:", AUX_DIR)

## Extract Initialization Values

The written auxiliary package is loaded independently, checked, and simulated. Its simulation outputs provide the initialization values to apply to the original dynamic package.

In [ ]:
AuxiliaryOMC = OMCSessionLocal()
omc_call(AuxiliaryOMC, f'loadFile("{MODELICA_PKG_PATH}")')
omc_call(AuxiliaryOMC, "loadModel(Complex)")
omc_call(AuxiliaryOMC, "loadModel(ModelicaServices)")
omc_call(AuxiliaryOMC, f'loadFile("{DYNAWO_PKG_PATH}")')
omc_call(AuxiliaryOMC, f'loadFile("{AUX_PACKAGE_FILE}")')

omc_call(AuxiliaryOMC, f"checkModel({AUX_ROOT_MODEL})", parsed=False)
print("Auxiliary package root model checked successfully.")

In [ ]:
# The package is already loaded in this session; passing the file would copy only package.mo
AuxiliarySystem = ModelicaSystemOMC(session=AuxiliaryOMC)
AuxiliarySystem.model(model_name=AUX_ROOT_MODEL)
AuxiliarySystem.simulate(resultfile=AUX_PACKAGE + "_res.mat")

initializable_by_model = {}
init_values_by_model = {}

for model in chain:
    components = components_by_model[model]
    initializable_components = get_initializable_components(components, INIT_MODEL_BY_COMPONENT)
    init_values_by_component = extract_all_initialization_values(
        AuxiliarySystem,
        components,
        INIT_MODEL_BY_COMPONENT,
    )

    initializable_by_model[model] = initializable_components
    init_values_by_model[model] = init_values_by_component

for model in chain:
    initialized_component_names = sorted(init_values_by_model[model])
    label = ", ".join(initialized_component_names) if initialized_component_names else "<none>"
    print(f"Extracted initialization values for {model}: {label}")

## Build Initialized Package

A copy of the validated original dynamic package is populated with the values extracted from the auxiliary simulation and saved as the initialized package.

In [ ]:
INITIALIZED_NAME_MAP = initialized_name_map(chain, INITIALIZED_PACKAGE)

# Drop any class left over from a previous run. A missing class is fine here.
send_expression(DynamicOMC, f"deleteClass({INITIALIZED_PACKAGE})")
omc_call(
    DynamicOMC,
    f'loadString("within ; package {INITIALIZED_PACKAGE} end {INITIALIZED_PACKAGE};")',
    parsed=False,
)

for model in chain:
    initialized_model = INITIALIZED_NAME_MAP[model]
    initialized_name = initialized_model.split(".")[-1]

    print(f"Creating initialized model {initialized_model} from {model}")
    omc_call(DynamicOMC, f'copyClass({model}, "{initialized_name}", {INITIALIZED_PACKAGE})')

    apply_initialization_modifiers(
        DynamicOMC,
        initialized_model,
        initializable_by_model[model],
        init_values_by_model[model],
        INIT_MODEL_BY_COMPONENT,
    )

if os.path.isdir(INITIALIZED_DIR):
    shutil.rmtree(INITIALIZED_DIR)
os.makedirs(INITIALIZED_DIR, exist_ok=True)

write_package_files(
    INITIALIZED_PACKAGE_FILE,
    INITIALIZED_ORDER_FILE,
    INITIALIZED_PACKAGE,
    package_class_names(chain, INITIALIZED_NAME_MAP),
)

save_initialized_package_classes(
    DynamicOMC,
    chain,
    INITIALIZED_NAME_MAP,
    INITIALIZED_DIR,
)

print("Wrote initialized package:", INITIALIZED_DIR)

## Validate Initialized Package

The generated initialized package is loaded independently and simulated to verify that the final output can run.

In [ ]:
InitializedOMC = OMCSessionLocal()
omc_call(InitializedOMC, f'loadFile("{MODELICA_PKG_PATH}")')
omc_call(InitializedOMC, "loadModel(Complex)")
omc_call(InitializedOMC, "loadModel(ModelicaServices)")
omc_call(InitializedOMC, f'loadFile("{DYNAWO_PKG_PATH}")')
omc_call(InitializedOMC, f'loadFile("{INITIALIZED_PACKAGE_FILE}")')

omc_call(InitializedOMC, f"checkModel({INITIALIZED_ROOT_MODEL})", parsed=False)
print("Initialized package root model checked successfully.")

InitializedSystem = ModelicaSystemOMC(session=InitializedOMC)
InitializedSystem.model(model_name=INITIALIZED_ROOT_MODEL)

simflags = simulation_flags_without_log_stats(InitializedOMC, INITIALIZED_ROOT_MODEL)
initialized_resultfile_prefix = INITIALIZED_PACKAGE

send_expression(
    InitializedOMC,
    f'simulate({INITIALIZED_ROOT_MODEL}, outputFormat="csv", '
    f'fileNamePrefix="{initialized_resultfile_prefix}", simflags="{simflags}")',
    parsed=False,
)

initialized_resultfile = os.path.join(
    str(InitializedSystem.getWorkDirectory()),
    initialized_resultfile_prefix + "_res.csv",
)
print("Initialized package simulated successfully.")

## Optional: Plot Simulation Result

The final cell is an optional example of visualizing a variable from the initialized-package simulation result.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Select a variable to visualize from the initialized simulation.
PLOT_VARIABLE = "Load2.terminal.V.re"

result_columns = pd.read_csv(initialized_resultfile, nrows=0).columns
if PLOT_VARIABLE not in result_columns:
    raise RuntimeError(f'PLOT_VARIABLE "{PLOT_VARIABLE}" is not a variable in the simulation result.')
initialized_df = pd.read_csv(initialized_resultfile, usecols=["time", PLOT_VARIABLE])

fig, ax = plt.subplots()
ax.plot(initialized_df["time"], initialized_df[PLOT_VARIABLE], label=PLOT_VARIABLE)
ax.legend(loc="lower right")
ax.set_title("Initialized package response")
ax.set_xlabel("Time (s)")
ax.set_ylabel(PLOT_VARIABLE)